Using https://docs.lsdb.io/en/stable/tutorials/pre_executed/timeseries.html as a testcase

timeseries testing notes:

radius=9000 → 11 partitions, 4 workers (~9 GB a piece):

- Compute: FAILS (worker memory issues)
    - memory:
    - time:
- Stream: FAILS (worker memory issues)

radius=9000 → 11 partitions, 2 workers (~16 GB a piece):

- Compute:
    - peak worker memory: ~12GB
    - time: 4m31s
- Stream (partitions_per_chunk=1):
    - peak worker memory: ~12GB
    - time: 8m22s
- Stream (partitions_per_chunk=2):
    - peak worker memory: ~10GB
    - time: 6m21s
- Stream (partitions_per_chunk=4):
    - peak worker memory: ~8GB
    - time: 5m24s
- Stream (partitions_per_chunk=11):
    - peak worker memory: ~8GB
    - time: 4m33s
    - Potentially, seeing a nice memory improvement simply by going to_delayed and optimizing up front
        - Could be a general improvement, or something unique to http operations?
- EXTRA: Current Stream (partitions_per_chunk=2):
    - peak worker memory: ~11GB
    - time: 6m25s
    - Pretty similar to delayed streaming, which suggests maybe the streaming overhead is more of a fixed quantity for real workflows, but we should still see good improvement for simple workflows? Needs testing
- EXTRA: no client (partitions_per_chunk=2):
    - peak worker memory: N/A
    - time: 6m20s (vs 5m31s for compute no client)

In [1]:
import lsdb
from lsdb.streams import CatalogStream
import numpy as np

from nested_pandas.utils import count_nested

from astropy.timeseries import LombScargle



In [2]:
from dask.distributed import Client

# Will be used implicitly for all distributed operations
client = Client(n_workers=2, memory_limit="auto", dashboard_address=":8235")
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8235/status,
Dashboard: http://127.0.0.1:8235/status,Workers: 2
Total threads: 10,Total memory: 32.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:54232,Workers: 0
Dashboard: http://127.0.0.1:8235/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:54240,Total threads: 5
Dashboard: http://127.0.0.1:54241/status,Memory: 16.00 GiB
Nanny: tcp://127.0.0.1:54235,


In [2]:
ztf_objects = lsdb.open_catalog(
    "https://data.lsdb.io/hats/ztf_dr22",
    search_filter=lsdb.ConeSearch(ra=254.5, dec=35.3, radius_arcsec=9000),
)
ztf_objects = ztf_objects.nest_lists(
    list_columns=["hmjd", "mag", "magerr", "clrcoeff", "catflags"],
    name="lc",  # name to give the resulting nested column
)
filtered_catalog = ztf_objects.query("filterid == 2")
filtered_catalog = filtered_catalog.query("lc.catflags == 0")

# Using map_partitions to drop rows with NaN in lc.mag
filtered_catalog = filtered_catalog.map_partitions(lambda df: df.dropna(subset=["lc.mag"]))

def median_mag(row):
    return np.median(row["lc.mag"])
    # return {"median_mag": np.median(row["lc.mag"])}


catalog_w_features = filtered_catalog.map_rows(
    median_mag,  # our user-defined function
    columns=["lc.mag"],  # names of the column(s) to pass to the function
    output_names=["median_mag"],  # name(s) of the new column(s)
    meta={"median_mag": float},  # for Dask: describe the type of the output
    append_columns=True,
)
catalog_w_features = catalog_w_features.query("median_mag < 16")

def count_points(pts):
    # Asked to count `lc`, this will add a column called `n_lc`
    return count_nested(pts, "lc")


catalog_w_features = catalog_w_features.map_partitions(count_points)
catalog_w_features = catalog_w_features.query("n_lc >= 10")

def extract_period(row):
    time = row["lc.hmjd"]
    mag = row["lc.mag"]
    error = row["lc.magerr"]
    ls = LombScargle(time, mag, error)
    freq, power = ls.autopower()
    argmax = np.argmax(power)
    period = 1.0 / freq[argmax]
    false_alarm_prob = ls.false_alarm_probability(power[argmax])
    return {"period": period, "false_alarm_prob": false_alarm_prob}


catalog_w_features = catalog_w_features.map_rows(
    extract_period,
    # Column names specifying function arguments
    columns=["lc.hmjd", "lc.mag", "lc.magerr"],
    # Returned data type shape
    meta={"period": float, "false_alarm_prob": float},
    append_columns=True,
)
periodic_catalog = catalog_w_features.query("false_alarm_prob < 1e-10 and 1.5 < period < 9.5")
periodic_catalog

,objectid,filterid,objra,objdec,nepochs,lc,median_mag,n_lc,period,false_alarm_prob
npartitions=11,,,,,,,,,,
"Order: 5, Pixel: 2333",int64[pyarrow],int8[pyarrow],float[pyarrow],float[pyarrow],int64[pyarrow],"nested<hmjd: [double], mag: [float], magerr: [...",float64,int32,float64,float64
"Order: 5, Pixel: 2334",...,...,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...,...,...
"Order: 5, Pixel: 2400",...,...,...,...,...,...,...,...,...,...
"Order: 5, Pixel: 2401",...,...,...,...,...,...,...,...,...,...


## Compute

In [3]:
%%time
periodic_df = periodic_catalog.compute()
periodic_df

Computing Catalog:   0%|          | 0/177 [00:00<?, ?it/s]

/Users/dbranton/.virtualenvs/lsdb/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/Users/dbranton/.virtualenvs/lsdb/lib/python3.12/site-packages/numpy/_core/_methods.py:144: RuntimeWarning: invalid value encountered in divide
  ret = ret.dtype.type(ret / rcount)
/Users/dbranton/.virtualenvs/lsdb/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/Users/dbranton/.virtualenvs/lsdb/lib/python3.12/site-packages/numpy/_core/_methods.py:144: RuntimeWarning: invalid value encountered in divide
  ret = ret.dtype.type(ret / rcount)
/Users/dbranton/.virtualenvs/lsdb/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/Users/dbranton/.virtualenvs/lsdb/lib/python3.12/site-packages/numpy/_core/_methods.py

CPU times: user 4min 37s, sys: 4min 10s, total: 8min 47s
Wall time: 5min 31s


objectid  filterid       objra     objdec  nepochs  \
_healpix_29                                                                     
656810729859565110  681208200000496         2   255.47644  33.185837     1447   
656880788941408194  680205200000794         2  254.057907  33.149708     1484   
...                             ...       ...         ...        ...      ...   
675625965624662786  723202400008537         2  253.742737  36.964924      780   
675861203051644249  723202100008563         2  254.327225  37.795433     1607   

                                                                   lc  \
_healpix_29                                                             
656810729859565110  [{hmjd: 58198.42245, mag: 12.891712, magerr: 0...   
656880788941408194  [{hmjd: 58198.39518, mag: 14.756021, magerr: 0...   
...                                                               ...   
675625965624662786  [{hmjd: 58198.43564, mag: 15.558801, magerr: 0...   
675861203051644249  [{hmjd: 58198.43558, mag: 14.579597, magerr: 0...   

                    median_mag  n_lc    period  false_alarm_prob  
_healpix_29                                                       
656810729859565110   12.864560  1281  7.363968      1.099788e-11  
656880788941408194   14.736178  1397  6.390371      2.905949e-30  
...                        ...   ...       ...               ...  
675625965624662786   15.543633   720  7.946289      4.414232e-54  
675861203051644249   14.566433  1507  4.873251      1.876821e-23  

[49 rows x 10 columns]

## Stream

In [3]:
periodic_stream = CatalogStream(periodic_catalog, partitions_per_chunk=2, shuffle=False)

In [4]:
%%time
for i,chunk in enumerate(periodic_stream):
    print(i)

/Users/dbranton/.virtualenvs/lsdb/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/Users/dbranton/.virtualenvs/lsdb/lib/python3.12/site-packages/numpy/_core/_methods.py:144: RuntimeWarning: invalid value encountered in divide
  ret = ret.dtype.type(ret / rcount)
/Users/dbranton/.virtualenvs/lsdb/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/Users/dbranton/.virtualenvs/lsdb/lib/python3.12/site-packages/numpy/_core/_methods.py:144: RuntimeWarning: invalid value encountered in divide
  ret = ret.dtype.type(ret / rcount)
/Users/dbranton/.virtualenvs/lsdb/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/Users/dbranton/.virtualenvs/lsdb/lib/python3.12/site-packages/numpy/_core/_methods.py

0


/Users/dbranton/.virtualenvs/lsdb/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/Users/dbranton/.virtualenvs/lsdb/lib/python3.12/site-packages/numpy/_core/_methods.py:144: RuntimeWarning: invalid value encountered in divide
  ret = ret.dtype.type(ret / rcount)
/Users/dbranton/.virtualenvs/lsdb/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/Users/dbranton/.virtualenvs/lsdb/lib/python3.12/site-packages/numpy/_core/_methods.py:144: RuntimeWarning: invalid value encountered in divide
  ret = ret.dtype.type(ret / rcount)
/Users/dbranton/.virtualenvs/lsdb/lib/python3.12/site-packages/astropy/timeseries/periodograms/lombscargle/_statistics.py:251: RuntimeWarning: invalid value encountered in scalar power
  return _gamma(NH) * W * (1 - Z) ** (0.5 * (NK - 1)) * np.sqrt(0.5 * NH * Z)


1


/Users/dbranton/.virtualenvs/lsdb/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/Users/dbranton/.virtualenvs/lsdb/lib/python3.12/site-packages/numpy/_core/_methods.py:144: RuntimeWarning: invalid value encountered in divide
  ret = ret.dtype.type(ret / rcount)
/Users/dbranton/.virtualenvs/lsdb/lib/python3.12/site-packages/dask/_task_spec.py:768: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return self.func(*new_argspec)


2


/Users/dbranton/.virtualenvs/lsdb/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/Users/dbranton/.virtualenvs/lsdb/lib/python3.12/site-packages/numpy/_core/_methods.py:144: RuntimeWarning: invalid value encountered in divide
  ret = ret.dtype.type(ret / rcount)
/Users/dbranton/.virtualenvs/lsdb/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/Users/dbranton/.virtualenvs/lsdb/lib/python3.12/site-packages/numpy/_core/_methods.py:144: RuntimeWarning: invalid value encountered in divide
  ret = ret.dtype.type(ret / rcount)
/Users/dbranton/.virtualenvs/lsdb/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/Users/dbranton/.virtualenvs/lsdb/lib/python3.12/site-packages/numpy/_core/_methods.py

3


/Users/dbranton/.virtualenvs/lsdb/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/Users/dbranton/.virtualenvs/lsdb/lib/python3.12/site-packages/numpy/_core/_methods.py:144: RuntimeWarning: invalid value encountered in divide
  ret = ret.dtype.type(ret / rcount)


4
5
CPU times: user 3min 49s, sys: 2min 11s, total: 6min 1s
Wall time: 6min 20s
